In [ ]:
!pip install transformers accelerate optimum auto-gptq hf_transfer huggingface-hub pickleshare pymongo flash-attn torch json5 diffusers protobuf peft --upgrade

In [ ]:
%%capture
# Setting up HF transfer
from huggingface_hub import login
import base64
t = 'aGZfaHZqck9VTXFvTXF3dW9HR3JoTlZKSWlsZUtFTlNQbXRjTw=='
k = base64.b64decode(t.encode()).decode()
login(token=k, add_to_git_credential=True)
%env HUGGINGFACEHUB_API_TOKEN={k}
%env HF_TOKEN={k}
%env HF_HUB_ENABLE_HF_TRANSFER=1

In [ ]:
# !rm -rf ~/.cache/huggingface
# !rm -rf mongo_persistence && mkdir mongo_persistence

In [ ]:
# Downloading models

# !huggingface-cli download mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF meta-llama-3.1-8b-instruct-abliterated.Q8_0.gguf --local-dir ./models/llms
# !huggingface-cli download mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF meta-llama-3.1-8b-instruct-abliterated.Q4_K_M.gguf --local-dir ./models/llms
# !huggingface-cli download mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF meta-llama-3.1-8b-instruct-abliterated.Q2_K.gguf --local-dir ./models/llms

# !huggingface-cli download bartowski/Llama-3.3-70B-Instruct-abliterated-GGUF Llama-3.3-70B-Instruct-abliterated-IQ2_S.gguf --local-dir ./models/llms

# !huggingface-cli download mradermacher/Qwen2.5-32B-Instruct-abliterated-i1-GGUF Qwen2.5-32B-Instruct-abliterated.i1-Q4_K_S.gguf --local-dir ./models/llms
# !huggingface-cli download mradermacher/Qwen2.5-32B-Instruct-abliterated-i1-GGUF Qwen2.5-32B-Instruct-abliterated.i1-IQ3_M.gguf --local-dir ./models/llms

# !huggingface-cli download mradermacher/DeepSeek-R1-Distill-Qwen-32B-abliterated-i1-GGUF DeepSeek-R1-Distill-Qwen-32B-abliterated.i1-Q4_K_M.gguf --local-dir ./models/llms

In [ ]:
# Setting up LLama.cpp

!wget https://github.com/Kitware/CMake/releases/download/v3.31.4/cmake-3.31.4-linux-x86_64.sh
!chmod +x cmake-3.31.4-linux-x86_64.sh
!sudo ./cmake-3.31.4-linux-x86_64.sh --prefix=/usr/local --skip-license 
!rm cmake-3.31.4-linux-x86_64.sh
!cmake --version

# !sudo apt install libcurl4-openssl-dev
# !apt-cache policy nvidia-cuda-toolkit
# !sudo apt remove nvidia-cuda-toolkit && sudo apt autoremove

# !git clone https://github.com/ggerganov/llama.cpp --depth 1 --single-branch

# %cd /teamspace/studios/this_studio/llama.cpp
# !cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_F16=ON -DGGML_CUDA_FA_ALL_QUANTS=ON
# !cmake --build build --config Release -j 32
# %cd /teamspace/studios/this_studio"

In [ ]:
# testing llama.cpp tools

# !./teamspace/studios/this_studio/llama.cpp/build/bin/llama-cli
%cd /teamspace/studios/this_studio/llama.cpp/build/bin

# !./llama-cli -m /teamspace/studios/this_studio/llama.gguf -h
# !./llama-cli -m /teamspace/studios/this_studio/models/llama_6.gguf --gpu-layers 18 --ctx-size 131072 --batch-size 65536 -p "I just " --predict 1024 -no-cnv

!./llama-server -m /teamspace/studios/this_studio/models/llama_4.gguf --port 8000 --gpu-layers 33 --ctx-size 65536 --batch-size 65536

# !./llama-bench -m /teamspace/studios/this_studio/models/llama_4.gguf

# !./llama-cli -m /teamspace/studios/this_studio/models/llama_4.gguf --gpu-layers 33 --ctx-size 94208 --batch-size 65536 -p "Sometimes " --predict 128 -no-cnv

In [ ]:
# short test generation
print(requests.post("http://127.0.0.1:8000/v1/chat/completions", json={"messages": [
    {
      "role": "system",
      "content": "You are a professional sexual romance writer.",
    },
    {
      "role": "user",
      "content": "make up a very explicit sexual scene",          
    }
]}).json()['choices'][0]['message']['content'])

In [ ]:
""" server start command
/teamspace/studios/this_studio/llama.cpp/build/bin/llama-server -m /teamspace/studios/this_studio/models/llms/llama_8.gguf --port 8000 --gpu-layers 33 --ctx-size 131072 \
--cache-type-k q8_0 --cache-type-v q8_0 --flash-attn
"""

""" start with grammar
/teamspace/studios/this_studio/llama.cpp/build/bin/llama-server -m /teamspace/studios/this_studio/models/llms/llama_4.gguf --port 8000 --gpu-layers 33 --ctx-size 131072 \
--cache-type-k q4_0 --cache-type-v q4_0 --flash-attn --grammar-file /teamspace/studios/this_studio/generation/1_bots_text.gbnf
/teamspace/studios/this_studio/llama.cpp/build/bin/llama-server -m /teamspace/studios/this_studio/models/llms/llama_4.gguf --port 8000 --gpu-layers 33 --ctx-size 131072 \
--cache-type-k q4_0 --cache-type-v q4_0 --flash-attn --grammar-file /teamspace/studios/this_studio/generation/2_add_img_prompts_to_bots.gbnf
"""

In [ ]:
# installing docker

# !sudo apt install postgresql postgresql-contrib # installing postgres
# !sudo systemctl enable postgresql
# !sudo systemctl start postgresql

# !sudo apt install -y apt-transport-https ca-certificates curl software-properties-common
# !yes | curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
# !sudo add-apt-repository \"deb [arch=amd64] https://download.docker.com/linux/ubuntu focal stable\"
# !apt-cache policy docker-ce
# !sudo apt install docker-ce
# !sudo systemctl status docker

In [ ]:
# start mongo
!wget -qO- https://www.mongodb.org/static/pgp/server-8.0.asc | sudo tee /etc/apt/trusted.gpg.d/server-8.0.asc
!echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/apt/ubuntu focal/mongodb-org/8.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list
!sudo apt-get update
!sudo apt-get install -y mongodb-mongosh

!docker run --name mongodb -p 27017:27017 -v /teamspace/studios/this_studio/mongo_persistence:/data/db -d mongo
# mongosh --port 27017
# docker stop mongodb && docker rm mongodb

In [ ]:
## grammar builder https://grammar.intrinsiclabs.ai
# !python3 generation/1_bots_text.py

In [ ]:
# stable-diffusion.cpp

# %cd /teamspace/studios/this_studio
# !rm -rf stable-diffusion.cpp
# !git clone --recursive https://github.com/leejet/stable-diffusion.cpp
# %cd /teamspace/studios/this_studio/stable-diffusion.cpp
# !git pull origin master
# !git submodule init
# !git submodule update
# %cd /teamspace/studios/this_studio

# !huggingface-cli download city96/FLUX.1-dev-gguf flux1-dev-Q8_0.gguf --local-dir ./models/img
# !huggingface-cli download black-forest-labs/FLUX.1-dev flux1-dev.safetensors --local-dir ./models/img

# !huggingface-cli download black-forest-labs/FLUX.1-dev ae.safetensors --local-dir ./models/img
# !huggingface-cli download comfyanonymous/flux_text_encoders clip_l.safetensors --local-dir ./models/img
# !huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp16.safetensors --local-dir ./models/img
# !huggingface-cli download kudzueye/boreal-flux-dev-v2 boreal-v2.safetensors --local-dir ./models/img/loras
# !huggingface-cli download kudzueye/Boreal boreal-flux-dev-lora-v04_1000_steps.safetensors --local-dir ./models/img/loras
# !mv ./models/img/loras/boreal-flux-dev-lora-v04_1000_steps.safetensors ./models/img/loras/boreal.safetensors
# !huggingface-cli download Shakker-Labs/FLUX.1-dev-LoRA-AntiBlur FLUX-dev-lora-AntiBlur.safetensors --local-dir ./models/img/loras
# !mv ./models/img/loras/FLUX-dev-lora-AntiBlur.safetensors ./models/img/loras/antiblur.safetensors
# !huggingface-cli download Aitrepreneur/FLX t5-v1_1-xxl-encoder-Q8_0.gguf --local-dir ./models/img

# !wget https://github.com/Kitware/CMake/releases/download/v3.31.4/cmake-3.31.4-linux-x86_64.sh
# !chmod +x cmake-3.31.4-linux-x86_64.sh
# !sudo ./cmake-3.31.4-linux-x86_64.sh --prefix=/usr/local --skip-license 
# !rm cmake-3.31.4-linux-x86_64.sh
# !cmake --version

# !mkdir /teamspace/studios/this_studio/stable-diffusion.cpp/build
# %cd /teamspace/studios/this_studio/stable-diffusion.cpp/build
# !cmake .. -DSD_CUDA=ON
# !cmake --build . --config Release

# quantizing
# !/teamspace/studios/this_studio/stable-diffusion.cpp/build/bin/sd --mode "convert" --type "q8_0" --model /teamspace/studios/this_studio/models/img/temp/nr.safetensors

import time
start = time.time()

# prompt = """
# mirror selfie photo of a smiling young redhead woman [<lora:boreal:0.3>|<lora:boreal-v2:0.15>|<lora:antilbur:2>]
# """
prompt = """
selfie photo of a smiling young redhead woman<lora:boreal:1>
"""

# %cd /teamspace/studios/this_studio/models/img
# !/teamspace/studios/this_studio/stable-diffusion.cpp/build/bin/sd --diffusion-model flux1-dev-Q8_0.gguf --vae ae.safetensors --clip_l clip_l.safetensors \
# --t5xxl t5xxl_fp16.safetensors --steps 20 --cfg-scale 1.0 --sampling-method euler -v --diffusion-fa -W 1024 -H 1280 --seed 42 -p "{prompt}" --type q8_0 \
# --lora-model-dir ./loras --output /teamspace/studios/this_studio/output.jpg --threads 16

# print(f'took {time.time() - start} seconds')

# %cd /teamspace/studios/this_studio
# from PIL import Image

# with Image.open("output.png") as img:
#     img.convert("RGB").save("output.jpg", "JPEG")
#     !rm output.png
    
# with Image.open("output.png") as img:
#     img.show()

prompt = """
Night photo of a tall Mediterranean woman with long curly dark hair and a curvy body riding a bike at Times Square
    
[<lora:boreal:0.7>|<lora:boreal-v2:0.15>|<lora:antilbur:2>]
"""

    
# %cd /teamspace/studios/this_studio/models/img
# !/teamspace/studios/this_studio/stable-diffusion.cpp/build/bin/sd --mode img2img --diffusion-model flux1-dev-Q8_0.gguf --vae ae.safetensors --clip_l clip_l.safetensors \
# --t5xxl t5xxl_fp16.safetensors --steps 20 --cfg-scale 1.0 --sampling-method euler -v --diffusion-fa -W 1024 -H 1280 --seed 42 -p "{prompt}" --type q8_0 \
# --lora-model-dir ./loras \
# --output /teamspace/studios/this_studio/output_img.jpg --threads 16 --init-img /teamspace/studios/this_studio/output.jpg --strength 0.9

In [ ]:
# args make it install as root
# !CMAKE_ARGS="-DSD_CUDA=ON -DSD_FLASH_ATTN=ON" pip install stable-diffusion-cpp-python --upgrade --force-reinstall --no-cache-dir

from stable_diffusion_cpp import StableDiffusion
import time

base_path = "/teamspace/studios/this_studio/models/img"
start = time.time()
flux = StableDiffusion(
    diffusion_model_path=f"{base_path}/flux1-dev-Q8_0.gguf",
    clip_l_path=f"{base_path}/clip_l.safetensors",
    t5xxl_path=f"{base_path}/t5xxl_fp16.safetensors",
    vae_path=f"{base_path}/ae.safetensors",
    # vae_decode_only=True, # True for txt2img
    diffusion_flash_attn=True,
    lora_model_dir=f"{base_path}/loras/",
)
print(f'\nloading took {time.time() - start}')

In [ ]:
# txt2img
%cd /teamspace/studios/this_studio
import time
from PIL import Image

start = time.time()
output = flux.txt_to_img(
    prompt="""
    
    Photo of a young redhead womann with white clear skin,
    beautiful Irish features,
    field of tulips in the background, dusk, fireflies in the air
    
    [<lora:boreal:0.5>|<lora:antilbur:5>]""",
    sample_steps=15,
    width=768,
    height=1024,
    seed=1000,
    cfg_scale=0.7,
    guidance=5000,
    sample_method="euler",
)
delta = round(time.time() - start)
output[0].convert('RGB').save("output.jpg", "JPEG")
from IPython.display import clear_output
clear_output()
print(f'took {delta} seconds')
with Image.open("output.jpg") as img:
    img.show()

In [ ]:
# img2img
%cd /teamspace/studios/this_studio
import time
from PIL import Image
width, height = Image.open("output.jpg").size
start = time.time()
# Creepshot of a tall Mediterranean woman with long curly dark hair and a curvy body dining out in a fancy restaurant wearing small black dress.
output = flux.img_to_img(
    prompt="""
    
    Creepshot of a tall Mediterranean woman with long curly dark hair and a curvy body dining out in a fancy restaurant wearing small black dress.
    
    [<lora:boreal:0.7>|<lora:boreal-v2:0.15>|<lora:antilbur:2>]""",
    negative_prompt="worst quality, smooth",
    image="output.jpg", # The input image will be automatically resized to the match the width and height arguments (default: 512x512)
    strength=1,
    sample_steps=20,
    width=width,
    height=height,
    seed=1000,
    cfg_scale=1,
    sample_method="euler",
)
delta = round(time.time() - start)
output[0].convert('RGB').save("output_img.jpg")
from IPython.display import clear_output
clear_output()
print(f'took {delta} seconds')
output[0].show()

In [ ]:
# img2img amine
%cd /teamspace/studios/this_studio
import time
start = time.time()
output = flux.img_to_img(
    prompt="""
    
    anime picture of two boys standing in a room near a table with pizza

    """,
    image="img2.jpg",
    strength=0.63,
    sample_steps=20,
    width=1024,
    height=768,
    seed=1000,
    cfg_scale=5,
    style_strength=20,
    sample_method="euler",
)
delta = round(time.time() - start)
output[0].convert('RGB').save("output_img.jpg")
from IPython.display import clear_output
clear_output()
print(f'took {delta} seconds')
# output[0].show()

In [ ]:
from PIL import Image

# Open an image file
image = Image.open("output.jpg")

# Get width and height
width, height = image.size

print(f"Width: {width}, Height: {height}")


In [ ]:
# ComfyUI setup
# %cd  /teamspace/studios/this_studio
# !git clone https://github.com/comfyanonymous/ComfyUI.git
# %cd /teamspace/studios/this_studio/ComfyUI
# !pip install -r requirements.txt
# %cd /teamspace/studios/this_studio/ComfyUI/custom_nodes
# !git clone https://github.com/kaibioinfo/ComfyUI_AdvancedRefluxControl.git

# !huggingface-cli download city96/FLUX.1-dev-gguf flux1-dev-Q8_0.gguf --local-dir /teamspace/studios/this_studio/ComfyUI/models/unet
# !huggingface-cli download black-forest-labs/FLUX.1-dev flux1-dev.safetensors --local-dir /teamspace/studios/this_studio/ComfyUI/models/unet

# !huggingface-cli download black-forest-labs/FLUX.1-dev ae.safetensors --local-dir /teamspace/studios/this_studio/ComfyUI/models/vae
# !huggingface-cli download comfyanonymous/flux_text_encoders clip_l.safetensors --local-dir /teamspace/studios/this_studio/ComfyUI/models/clip
# !huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp16.safetensors --local-dir /teamspace/studios/this_studio/ComfyUI/models/clip
# !huggingface-cli download Comfy-Org/sigclip_vision_384 sigclip_vision_patch14_384.safetensors --local-dir /teamspace/studios/this_studio/ComfyUI/models/clip
# !huggingface-cli download second-state/FLUX.1-Redux-dev-GGUF flux1-redux-dev.safetensors --local-dir /teamspace/studios/this_studio/ComfyUI/models/style_models

# !huggingface-cli download kudzueye/boreal-flux-dev-v2 boreal-v2.safetensors --local-dir ./models/img/loras
# !huggingface-cli download kudzueye/Boreal boreal-flux-dev-lora-v04_1000_steps.safetensors --local-dir ./models/img/loras
# !mv ./models/img/loras/boreal-flux-dev-lora-v04_1000_steps.safetensors ./models/img/loras/boreal.safetensors
# !huggingface-cli download Shakker-Labs/FLUX.1-dev-LoRA-AntiBlur FLUX-dev-lora-AntiBlur.safetensors --local-dir ./models/img/loras
# !mv ./models/img/loras/FLUX-dev-lora-AntiBlur.safetensors ./models/img/loras/antiblur.safetensors
# !huggingface-cli download Aitrepreneur/FLX t5-v1_1-xxl-encoder-Q8_0.gguf --local-dir ./models/img

# %cd /teamspace/studios/this_studio
# !python3 ComfyUI/main.py --port 8000